In [17]:
"""
Updated on Mon Oct 1  16:00:00 2025
Created on Mon Sep 23 17:03:01 2021
@author: Oumbeg
"""

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import re
from time import sleep
import os
import requests
import pdfplumber

    
    
    
    

In [18]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'AG FSRCAG'
print(f"Running {regulatorName} Web Scraping Tool v.1.0")
now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)





Running AG FSRCAG Web Scraping Tool v.1.0


In [19]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------


chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory" : tempfolder, 
        "plugins.always_open_pdf_externally": True}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}



In [20]:


#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

def clean_list(lst):
    cleaned = []
    for i, item in enumerate(lst):
        # Skip if it's a prefix of the previous item
        if i > 0 and lst[i - 1].startswith(item):
            continue
        cleaned.append(item)
    return cleaned

def findTel(mydata):
    pattern = re.compile('[(+0-9)]+')
    matcher = pattern.findall(mydata)
    listToStr = ' '.join([str(elem) for elem in matcher])
    return listToStr


def findUrl(mydata):
    regex = re.compile(r"(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:'\".,<>?«»“”‘’]))")
    url = regex.findall(mydata)
    return ' '.join([str(elem) for elem in [x[0] for x in url]])

def findEmail(myData):
    """
    This function finds email in a string.
    :param myData: string
    :return: String
    """
    regex = re.compile('[a-zA-Z0-9.\-_]+@[^@]+\.[^@]+')
    email = regex.findall(myData)
    return ' '.join([str(elem) for elem in email])


def extract_field_text(field, text):
    # Create a regex pattern using the provided field name
    pattern = rf'{re.escape(field)}:?\s*(.*)'
    match = re.search(pattern, text)

    if match:
        return match.group(1).strip()
    return ''


	
# try to create an empty folder "tempfolder"
try:
    os.mkdir(tempfolder)
except:
    prevfiles=os.listdir(tempfolder)
    os.chdir(tempfolder)
    for prf in prevfiles:
        os.remove(prf)
    print('The directory tempfolder already exists.')
os.chdir(tempfolder)##only if files are going to be downloaded here

processdate=now.strftime('%Y-%m-%d')

patternTel = re.compile('(el.:|el\. |Tel:|Cell:|Tel.)')
patternFax = re.compile('(Fax.:|Fax:|Fax.|Fax)')


The directory tempfolder already exists.


<>:49: SyntaxWarning: invalid escape sequence '\-'
<>:78: SyntaxWarning: invalid escape sequence '\.'
<>:49: SyntaxWarning: invalid escape sequence '\-'
<>:78: SyntaxWarning: invalid escape sequence '\.'
C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_1728\2675396213.py:49: SyntaxWarning: invalid escape sequence '\-'
  regex = re.compile('[a-zA-Z0-9.\-_]+@[^@]+\.[^@]+')
C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_1728\2675396213.py:78: SyntaxWarning: invalid escape sequence '\.'
  patternTel = re.compile('(el.:|el\. |Tel:|Cell:|Tel.)')


In [ ]:
# %%

#------------------------------------------------ Main_Function ----------------------------------------
regdict={'AG FSRCAG 1': 'https://www.fsrc.gov.ag/index.php/directories'}
for reg in regdict:
    
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(3)
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    bloc = soup.find("div",{"class":"left"})
    url = bloc.find_all('a', href=True)
    
    for i in range(len(url)):
    #for i in range(1, 8):
        # download the PDFs from search using xpath

        #driver.find_element(By.XPATH, '//*[@id="content"]/div[2]/div/div/div/div[1]/a[%d]'%i).click()
        driver.get('https://www.fsrc.gov.ag'+ url[i]['href'])
        print('https://www.fsrc.gov.ag'+ url[i]['href'])
        sleep(3)
        while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:
            print('Waiting for file to download')
            sleep(3)
            break
        
            
        # Credit Unions    
        if i == 0:
            tables = []


            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)     
            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    table = page.extract_table()
                    tables.append(table)
            

            
            for table in tables:
                for index,row in enumerate(table):
                    # Check if any non-empty value exists in the row
                    row_values = [str(cell) for cell in row if cell is not None]
                    if any(row_values):

                        email_ = ''
                        tel_ = ''
                        website_ = ''
                        
                        if row_values[0] == '':
                            continue
                        else:
                            row_values[0].replace('\n',' ')
                        #print(index)
                        if len(row_values)>2:
                            #print(row_values)
                            clean_row_values = clean_list(row_values)
                            name = clean_row_values[0].replace('\n',' ')
                            # print('Address:',clean_row_values[1])
                            # print('MembershipType:', clean_row_values[-1])

                            sqldict['Name'].append(name)
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['ListCode'].append('7')
                            sqldict['RegCode'].append('FSRCAG')
                            sqldict['RegCtry'].append('AG')
                            sqldict['RegulationType'].append('Regulated')
                            sqldict['Address_1'].append(clean_row_values[1])
                            sqldict['License_Type'].append(clean_row_values[-1])
                            sqldict['ListName'].append('Credit Unions')
                        else:
                            
                            if 'Email' in row_values[0]:
                                email_ = row_values[0].split(' ')[-1]
                                
                            elif 'Tel' in row_values[0]:
                                tel_ = row_values[0].split('Tel')[-1].strip().removeprefix('.:')
                                if len(sqldict['Name']) > len(sqldict['Phone']):
                                    sqldict['Phone'].append('')
                                # else:
                                #     sqldict['Phone'].append(tel_.strip())
                            elif 'Website' in row_values[0]:
                                website_ = row_values[0].split('Website')[-1].strip().removeprefix(':')
                                if len(sqldict['Name']) > len(sqldict['Website']):
                                    sqldict['Website'].append('')
                                # else:
                                #     sqldict['Website'].append(website_.strip())        
                        sqldict = bourange_same_length_array(sqldict)           
                        
                        
                                
            os.remove(filePath)
        
        # CMT Service Providers
        elif i == 1:
            
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = []
            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    table = page.extract_table()
                    tables.append(table)

            for table in tables:
                for index,row in enumerate(table):
                    # Check if any non-empty value exists in the row
                    row_values = [str(cell) for cell in row if cell is not None]
                    if any(row_values):

                        if len(row_values)>2:
                                        #print(row_values)
                            clean_row_values = clean_list(row_values)

                            if len(clean_row_values)>2:
                                
                                if clean_row_values[1].replace('\n',' ') != 'NAME OF INSTITUTION':
                                    #print('Name:', clean_row_values[1].replace('\n',' '))
                                    name =  clean_row_values[1].replace('\n',' ')
                                    status_ = clean_row_values[2].replace('\n',' ')

                                    #print(clean_row_values[-1])
                                    email_ = extract_field_text('Email',clean_row_values[-1])
                                    tel_ = extract_field_text('Tel',clean_row_values[-1])
                                    fax_ = extract_field_text('Fax',clean_row_values[-1])
                                    website_ = extract_field_text('Website',clean_row_values[-1])

                                    address_ = clean_row_values[-1]
                                    real_address_ = re.sub(r'(Email:.*|Tel:.*|Fax:.*|Website:.*)\n?', '', address_).strip().replace('\n',' ')
                                    #print(real_address_)
                                    sqldict['Name'].append(name)
                                    sqldict['ListProcessDate'].append(processdate)
                                    sqldict['ListCode'].append('1')
                                    sqldict['RegCode'].append('FSRCAG')
                                    sqldict['RegCtry'].append('AG')
                                    sqldict['RegulationType'].append('Regulated')
                                    if len(real_address_)<3:
                                        sqldict['Address_1'].append('')
                                    else:
                                        sqldict['Address_1'].append(real_address_)
                                    sqldict['License_Type'].append(status_)
                                    sqldict['Email'].append(email_)
                                    sqldict['Phone'].append(tel_.split(' ')[0])
                                    if len(fax_)<5:
                                        sqldict['Fax'].append('')
                                    else:
                                        sqldict['Fax'].append(fax_)
                                    sqldict['Website'].append(website_)
                                    sqldict['ListName'].append('Listing of Corporate Management and Trust Service Providers in Antigua & Barbuda')
                                    sqldict = bourange_same_length_array(sqldict)
            os.remove(filePath)                                    
                                    

            
        # Banks   
        elif i == 2:
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = []
            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    table = page.extract_table()
                    tables.append(table)

            for table in tables:
                for index,row in enumerate(table):
                    # Check if any non-empty value exists in the row
                    row_values = [str(cell) for cell in row if cell is not None]
                    if any(row_values):

                        if len(row_values)>2:
                                        #print(row_values)
                            clean_row_values = clean_list(row_values)

                            if len(clean_row_values)>2:
                                
                                if clean_row_values[0].replace('\n',' ') != 'NAME OF INSTITUTION':
                                    name = clean_row_values[0].replace('\n',' ')
                                    #print('Name:', clean_row_values[0].replace('\n',' '))
                                    status_ = clean_row_values[1].replace('\n',' ')
                                    #print(clean_row_values[-1])
                                    email_ = extract_field_text('Email',clean_row_values[-1])
                                    tel_ = extract_field_text('Tel',clean_row_values[-1])
                                    fax_ = extract_field_text('Fax',clean_row_values[-1])
                                    address_ = clean_row_values[-1]
                                    website_ = extract_field_text('Website',clean_row_values[-1])
                           
                                    
                                    real_address_ = re.sub(r'(Email:.*|Tel:.*|Fax:.*|Website:.*)\n?', '', address_).strip().replace('\n',' ')
                                    sqldict['Name'].append(name)
                                    sqldict['ListProcessDate'].append(processdate)
                                    sqldict['ListCode'].append('2')
                                    sqldict['RegCode'].append('FSRCAG')
                                    sqldict['RegCtry'].append('AG')
                                    sqldict['RegulationType'].append('Regulated')
                                    if len(real_address_)<3:
                                        sqldict['Address_1'].append('')
                                    else:
                                        sqldict['Address_1'].append(real_address_)
                                    sqldict['License_Type'].append(status_)
                                    sqldict['Email'].append(email_)
                                    sqldict['Phone'].append(tel_.split(' ')[0])
                                    if len(fax_)<5:
                                        sqldict['Fax'].append('')
                                    else:
                                        sqldict['Fax'].append(fax_)
                                    sqldict['Website'].append(website_)
                                    sqldict['ListName'].append('Banks')
                                    sqldict = bourange_same_length_array(sqldict)
                                            
            os.remove(filePath)
            
        # Insurance Agents    
        elif i == 3:
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = []
            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    table = page.extract_table()
                    tables.append(table)
            for table in tables:
                for i in range(len(table)):

                    #print(table[i][0])
                    for j in range(len(table[i])):
                        name = table[i][j].split('\n')[0]
                        address = table[i][j].split('\n')[1:]
                        if name.strip() != '':
                            
                            # print(address)
                            if name.strip() != '':
                                address_ = ' '.join(address)

                            email_ = extract_field_text('Email',address_)
                            tel_ = extract_field_text('Tel',address_)
                            fax_ = extract_field_text('Fax',address_)
                            website_ = extract_field_text('Website',address_)
                            agent_for = extract_field_text('Agent',address_)

                            real_address_ = re.sub(r'(Email:.*|Tel:.*|Fax:.*|Website:.*|agent_for:.*)\n?', '', address_).strip().replace('\n',' ')
                            # print(real_address_)
                            sqldict['Name'].append(name)
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['ListCode'].append('3')
                            sqldict['RegCode'].append('FSRCAG')
                            sqldict['RegCtry'].append('AG')
                            sqldict['RegulationType'].append('Regulated')
                            if len(real_address_)<3:
                                sqldict['Address_1'].append('')
                            else:
                                sqldict['Address_1'].append(real_address_)
                            sqldict['License_Type'].append(status_)
                            sqldict['Email'].append(email_)
                            sqldict['Phone'].append(tel_.split(' ')[0])
                            if len(fax_)<5:
                                sqldict['Fax'].append('')
                            else:
                                sqldict['Fax'].append(fax_.split(' ')[0])
                            sqldict['Website'].append(website_)
                            sqldict['ListName'].append('Insurance Agents')
                            sqldict = bourange_same_length_array(sqldict)
            os.remove(filePath)    
            
        # Insurance Brokers    
        elif i == 4:
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = []
            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    table = page.extract_table()
                    tables.append(table)
            for table in tables:
                for i in range(len(table)):

                    #print(table[i][0])
                    for j in range(len(table[i])):
                        name = table[i][j].split('\n')[0]
                        address = table[i][j].split('\n')[1:]
                        if name.strip() != '':
                            
                            # print(address)
                            if name.strip() != '':
                                address_ = ' '.join(address)

                            email_ = extract_field_text('Email',address_)
                            tel_ = extract_field_text('Tel',address_)
                            fax_ = extract_field_text('Fax',address_)
                            website_ = extract_field_text('Website',address_)
                            agent_for = extract_field_text('Agent',address_)

                            real_address_ = re.sub(r'(Email:.*|Tel:.*|Fax:.*|Website:.*|agent_for:.*)\n?', '', address_).strip().replace('\n',' ')
                            # print(real_address_)
                            sqldict['Name'].append(name)
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['ListCode'].append('4')
                            sqldict['RegCode'].append('FSRCAG')
                            sqldict['RegCtry'].append('AG')
                            sqldict['RegulationType'].append('Regulated')
                            if len(real_address_)<3:
                                sqldict['Address_1'].append('')
                            else:
                                sqldict['Address_1'].append(real_address_)
                            sqldict['License_Type'].append(status_)
                            sqldict['Email'].append(email_)
                            sqldict['Phone'].append(tel_.split(' ')[0])
                            if len(fax_)<5:
                                sqldict['Fax'].append('')
                            else:
                                sqldict['Fax'].append(fax_)
                            sqldict['Website'].append(website_)
                            sqldict['ListName'].append('Insurance Brokers')
                            sqldict = bourange_same_length_array(sqldict)
            os.remove(filePath) 
            
        # Insurance Companies    
        elif i == 5:
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = []
            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    table = page.extract_table()
                    tables.append(table)
            for table in tables:
                for i in range(len(table)):

                    #print(table[i][0])
                    for j in range(len(table[i])):
                        name = table[i][j].split('\n')[0]
                        address = table[i][j].split('\n')[1:]
                        if name.strip() != '':
                            
                            # print(address)
                            if name.strip() != '':
                                address_ = ' '.join(address)

                            email_ = extract_field_text('Email',address_)
                            tel_ = extract_field_text('Tel',address_)
                            fax_ = extract_field_text('Fax',address_)
                            website_ = extract_field_text('Website',address_)
                            agent_for = extract_field_text('Agent',address_)

                            real_address_ = re.sub(r'(Email:.*|Tel:.*|Fax:.*|Website:.*|agent_for:.*)\n?', '', address_).strip().replace('\n',' ')
                            # print(real_address_)
                            sqldict['Name'].append(name)
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['ListCode'].append('5')
                            sqldict['RegCode'].append('FSRCAG')
                            sqldict['RegCtry'].append('AG')
                            sqldict['RegulationType'].append('Regulated')
                            if len(real_address_)<3:
                                sqldict['Address_1'].append('')
                            else:
                                sqldict['Address_1'].append(real_address_)
                            sqldict['License_Type'].append(status_)
                            sqldict['Email'].append(email_)
                            sqldict['Phone'].append(tel_.split(' ')[0])
                            if len(fax_)<5:
                                sqldict['Fax'].append('')
                            else:
                                sqldict['Fax'].append(fax_.split(' ')[0])
                            sqldict['Website'].append(website_)
                            sqldict['ListName'].append('Insurance Companies')
                            sqldict = bourange_same_length_array(sqldict)
            os.remove(filePath)            

            
        # Money Sevices Businesses    
        elif i == 6:
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = []
            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    table = page.extract_table()
                    tables.append(table)
            for table in tables:
                for index,row in enumerate(table):
                    # Check if any non-empty value exists in the row
                    row_values = [str(cell) for cell in row if cell is not None]
                    if any(row_values):

                        if len(row_values)>2:
                                        #print(row_values)
                            clean_row_values = clean_list(row_values)
                            

                            if len(clean_row_values)>2:
                                if clean_row_values[0].replace('\n',' ') != 'NAME OF INSTITUTION' and clean_row_values[0]!='':
                                    #print(clean_row_values)
                                    name = clean_row_values[0].replace('\n',' ')
                                    #print('Name:',name)
                                    status_ = clean_row_values[1].replace('\n',' ')
                                    #print(clean_row_values[-1])
                                    email_ = extract_field_text('Email',clean_row_values[-1])
                                    tel_ = extract_field_text('Tel',clean_row_values[-1])
                                    fax_ = extract_field_text('Fax',clean_row_values[-1])
                                    address_ = clean_row_values[-1]
                                    website_ = extract_field_text('Website',clean_row_values[-1])        
                                    real_address_ = re.sub(r'(Email:.*|Tel:.*|Fax:.*|Website:.*)\n?', '', address_).strip().replace('\n',' ')


                                elif clean_row_values[0]=='':
                                    if clean_row_values[1].replace('\n',' ') != 'NAME OF INSTITUTION' :
                                        name = clean_row_values[1].replace('\n',' ')
                                        status_ = clean_row_values[2].replace('\n',' ')
                                        #print(clean_row_values[-1])
                                        email_ = extract_field_text('Email',clean_row_values[-1])
                                        tel_ = extract_field_text('Tel',clean_row_values[-1])
                                        fax_ = extract_field_text('Fax',clean_row_values[-1])
                                        address_ = clean_row_values[-1]
                                        website_ = extract_field_text('Website',clean_row_values[-1])        
                                        real_address_ = re.sub(r'(Email:.*|Tel:.*|Fax:.*|Website:.*)\n?', '', address_).strip().replace('\n',' ')


                                sqldict['Name'].append(name)
                                sqldict['ListProcessDate'].append(processdate)
                                sqldict['ListCode'].append('8')
                                sqldict['RegCode'].append('FSRCAG')
                                sqldict['RegCtry'].append('AG')
                                sqldict['RegulationType'].append('Regulated')
                                if len(real_address_)<3:
                                    sqldict['Address_1'].append('')
                                else:
                                    sqldict['Address_1'].append(real_address_)
                                sqldict['License_Type'].append(status_)
                                # if len(tel_.split(' ')[0])<5:
                                #     sqldict['Phone'].append('')
                                # else:
                                #     sqldict['Phone'].append(tel_.split(' ')[0])

                                if len(fax_)<5:
                                    sqldict['Fax'].append('')
                                else:
                                    sqldict['Fax'].append(fax_)

                                sqldict['Website'].append(website_)
                                sqldict['ListName'].append('Money Service Businessess')
                                sqldict = bourange_same_length_array(sqldict)
            os.remove(filePath)
        
        # Digital Asset Business
        # elif i == 8:
        #     print(os.listdir(tempfolder))
        #     pdf_file = os.listdir(tempfolder)[0]
        #     filePath = os.path.join(tempfolder, pdf_file)
        
        #     tables = []
        #     with pdfplumber.open(filePath) as pdf:
        #         for page in pdf.pages:
        #             lines = page.extract_text_lines()
        #             total_address = []
        #             for line in lines:
        #                 name = ''.join([line['chars'][k]['text'] for k in range(len(line['chars'])) if 'Italic' not in line['chars'][k]['fontname'] and line['chars'][k]['x1'] < 240])
        #                 address = ''.join([line['chars'][k]['text'] for k in range(len(line['chars'])) if line['chars'][k]['x1'] > 410 and line['chars'][k]['y1'] > 210])

        #                 # Skip lines without a valid name or address
        #                 if not name.strip() or 'Limited' not in name:
        #                     continue
        #                 if 'mmission' in address or 'Address' in address:
        #                     continue

        #                 # Clean up the name and address
        #                 name = name.replace('\n', ' ').strip()
        #                 address = address.replace('\n', ' ').strip()

        #                 name = re.sub(r'(?<!\s)(L)', r' \1', name)

        #                 # Print the name and address in the desired format
        #                 # print(f"Name: {name.replace(',','')}")
        #                 # print(f"Address: {address.replace(',','')}")
        #                 sqldict['Name'].append(name.replace(',',''))
        #                 sqldict['ListProcessDate'].append(processdate)
        #                 sqldict['ListCode'].append('9')
        #                 sqldict['RegCode'].append('FSRCAG')
        #                 sqldict['RegCtry'].append('AG')
        #                 sqldict['RegulationType'].append('Regulated')
        #                 sqldict['Address_1'].append(address.replace(',',''))
        #                 sqldict['ListName'].append('Digital Asset Businesses')

        #                 sqldict = bourange_same_length_array(sqldict)

        elif i == 8:
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)

            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    # Group characters by line using y0 position
                    lines_dict = {}
                    for char in page.chars:
                        y0 = round(char['y0'], 1)
                        lines_dict.setdefault(y0, []).append(char)

                    for y0 in sorted(lines_dict.keys()):
                        line_chars = lines_dict[y0]

                        # Extract name from non-italic text on the left
                        name = ''.join([
                            c['text'] for c in line_chars
                            if 'Italic' not in c['fontname'] and c['x1'] < 240
                        ])

                        # Extract address from right-side characters
                        address = ''.join([
                            c['text'] for c in line_chars
                            if c['x1'] > 410 and c['y1'] > 210
                        ])

                        # Skip invalid entries
                        if not name.strip() or 'Limited' not in name:
                            continue
                        if 'mmission' in address or 'Address' in address:
                            continue

                        # Clean and format
                        
                        address = address.replace('\n', ' ').strip()

                        # Append to sqldict
                        sqldict['Name'].append(name.replace(',', ''))
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append('9')
                        sqldict['RegCode'].append('FSRCAG')
                        sqldict['RegCtry'].append('AG')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['Address_1'].append(address.replace(',', ''))
                        sqldict['ListName'].append('Digital Asset Businesses')

                        sqldict = bourange_same_length_array(sqldict)





Working with AG FSRCAG 1
https://www.fsrc.gov.ag/images/pdf/Directory_of_Credit_Unions.pdf
['Directory_of_Credit_Unions.pdf']
https://www.fsrc.gov.ag/images/pdf/ibc_forms/Licensed_CMTSPs-Latest.pdf
['Licensed_CMTSPs-Latest.pdf']
https://www.fsrc.gov.ag/images/pdf/banking/directory_of_banks_website_version-latest.pdf
['directory_of_banks_website_version-latest.pdf']
https://www.fsrc.gov.ag/images/pdf/insurance/Insurance_agents.pdf
['Insurance_agents.pdf']
https://www.fsrc.gov.ag/images/pdf/insurance/Insurance_brokers.pdf
['Insurance_brokers.pdf']
https://www.fsrc.gov.ag/images/pdf/insurance/Insurance_companies.pdf
['Insurance_companies.pdf']
https://www.fsrc.gov.ag/images/pdf/Directory_of_Licensed_Money_Service_Businesses.pdf
['Directory_of_Licensed_Money_Service_Businesses.pdf']
https://www.fsrc.gov.ag#
Waiting for file to download
https://www.fsrc.gov.ag/images/pdf/digital-assets/Directory_of_Licenced_Digital_Asset_Business.pdf
['Directory_of_Licenced_Digital_Asset_Business.pdf']


In [24]:

# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------


os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

#df  = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
     

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_1728\3041553946.py:12: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df['Phone'] = df['Phone'].apply(lambda x: '' if len(str(x)) < 5 else x)

In [25]:

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 90 values.
Key 'priority' has 90 values.
Key 'ListLabel' has 90 values.
Key 'Typology' has 90 values.
Key 'EntryType' has 90 values.
Key 'Name' has 90 values.
Key 'InternalID_1' has 90 values.
Key 'InternalID_1_type' has 90 values.
Key 'InternalID_2' has 90 values.
Key 'InternalID_2_type' has 90 values.
Key 'InternalID_3' has 90 values.
Key 'InternalID_3_type' has 90 values.
Key 'CoType' has 90 values.
Key 'License_Type' has 90 values.
Key 'Address_1' has 90 values.
Key 'Address_2' has 90 values.
Key 'City' has 90 values.
Key 'Zip' has 90 values.
Key 'Cntry' has 90 values.
Key 'Phone' has 90 values.
Key 'Fax' has 90 values.
Key 'Website' has 90 values.
Key 'Email' has 90 values.
Key 'RegulationType' has 90 values.
Key 'RegulationTypeCode' has 90 values.
Key 'RegulationDate' has 90 values.
Key 'CancellationDate' has 90 values.
Key 'RegCtry' has 90 values.
Key 'RegCode' has 90 values.
Key 'ListCode' has 90 values.
Key 'ListLanguage' has 90 values.
Key 'ListValidityDate' h

In [21]:
df.to_csv('list_3.csv')

In [ ]:
df.to_csv('FSRCAG_total_v5{}.csv'.format(now.strftime('%Y%m%d%h')), index=False)

: 

In [9]:
tables = []
with pdfplumber.open(filePath) as pdf:
    for page in pdf.pages:
        table = page.extract_table()
        tables.append(table)

for table in tables:
    for index,row in enumerate(table):
        # Check if any non-empty value exists in the row
        row_values = [str(cell) for cell in row if cell is not None]
        if any(row_values):

            if len(row_values)>1:
                            #print(row_values)
                clean_row_values = clean_list(row_values)

                if len(clean_row_values)>2:
                    
                    if clean_row_values[0].replace('\n',' ') != 'NAME OF INSTITUTION':
                        print('Name:', clean_row_values[0].replace('\n',' '))
                        print(clean_row_values[0])
                        email_ = extract_field_text('Email',clean_row_values[-1])
                        tel_ = extract_field_text('Tel',clean_row_values[-1])
                        fax_ = extract_field_text('Fax',clean_row_values[-1])
                        address_ = clean_row_values[-1]
                        
                        
                        
                        real_address_ = re.sub(r'(Email:.*|Tel:.*|Fax:.*|Website:.*)\n?', '', address_).strip().replace('\n',' ')
                        #print(real_address_)
                        # print(email_)
                        # print(tel_.split(';')[0])
                        # print(Fax_)

FileNotFoundError: [Errno 2] No such file or directory: "C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\AG FSRCAG\\tempfolder\\afaf254c-ec5c-4e28-b2d8-84d46b565397.tmp"

In [18]:
print('https://www.fsrc.gov.ag/'+ url[len(url)-1]['href'])

https://www.fsrc.gov.ag//images/pdf/digital-assets/Directory_of_Licenced_Digital_Asset_Business.pdf


In [19]:
print(len(url))

9


In [ ]:
for table in tables:
    for index,row in enumerate(table):
        # Check if any non-empty value exists in the row
        row_values = [str(cell) for cell in row if cell is not None]
        if any(row_values):

            print(row_values)

            if len(row_values)>2:
                            #print(row_values)
                clean_row_values = clean_list(row_values)

                if len(clean_row_values)>2:
                    
                    if clean_row_values[1].replace('\n',' ') != 'NAME OF INSTITUTION':
                        print('Name:', clean_row_values[1].replace('\n',' '))
                        #print(clean_row_values[-1])
                        email_ = extract_field_text('Email',clean_row_values[-1])
                        tel_ = extract_field_text('Tel',clean_row_values[-1])
                        fax_ = extract_field_text('Fax',clean_row_values[-1])
                        address_ = clean_row_values[-1]
                        
                        
                        
                        real_address_ = re.sub(r'(Email:.*|Tel:.*|Fax:.*)\n?', '', address_).strip().replace('\n',' ')
                        print(real_address_)
                        # print(email_)
                        # print(tel_.split(';')[0])
                        # print(Fax_)
                        

["ANJO INSURANCES AGENCY\nWoods Centre\nP.O. Box 104,\nSt. John's, Antigua\nTel: 480-3050 | Fax: 480-3064\nAgent for:\nCG United Insurance Ltd.\nSagicor Life (Eastern Caribbean) Inc.", "BRYSON’S INSURANCE AGENCY\nFriar's Hill Road\nP.O. Box 162,\nSt. John’s, Antigua\nTel: 480-1220 | Fax: 462-5538\nAgent for:\nNational General Insurance Co. Ltd. (NAGICO)\nCUNA Caribbean Insurance OECS Limited\nLloyds Underwriters"]
['CIBC FIRST CARIBBEAN INSURANCE AGENT\nC/o CIBC First Caribbean International Bank\nLtd.\nOld Parham Road Branch,\nSt. John’s, Antigua.\nTel: 480-5050 | Fax: 480-5140\nAgent for:\nCG United Insurance Ltd.', "KENNETH A. GOMEZ INSURANCE AGENCY LTD.\n(Formally Kenneth A, Gomez & Sons Insurance\nAgency)\nRoyal Palm Place, Friars Hill Road,\nSt. John's, Antigua\nTel: 481-1850 | Fax: 481-1859\nAgent for:\nGuardian General Insurance Ltd.\nGuardian Life (OECS) Ltd.\nNetherlands Insurance Co. Ltd."]
["SALIENT INSURANCE AGENCY LTD.\nUnit 10 & 11, Mandolin Place,\nFriars Hill Road, St.

In [ ]:
for table in tables:
    for i in range(len(table)):

        #print(table[i][0])
        for j in range(len(table[i])):
            name = table[i][j].split('\n')[0]
            address = table[i][j].split('\n')[1:]
            if name.strip() != '':
                print(name.replace('\n',' '))
                # print(address)
                if name.strip() != '':
                    address_ = ' '.join(address)

                email_ = extract_field_text('Email',address_)
                tel_ = extract_field_text('Tel',address_)
                fax_ = extract_field_text('Fax',address_)
                website_ = extract_field_text('Website',address_)
                agent_for = extract_field_text('Agent',address_)

                real_address_ = re.sub(r'(Email:.*|Tel:.*|Fax:.*|Website:.*|agent_for:.*)\n?', '', address_).strip().replace('\n',' ')
                print(real_address_)
                # sqldict['Name'].append(name)
                # sqldict['ListProcessDate'].append(processdate)
                # sqldict['ListCode'].append('3')
                # sqldict['RegCode'].append('FSRCAG')
                # sqldict['RegCtry'].append('AG')
                # sqldict['RegulationType'].append('Regulated')
                # sqldict['Address_1'].append(real_address_)
                # sqldict['License_Type'].append(status_)
                # sqldict['Email'].append(email_)
                # sqldict['Phone'].append(tel_)
                # sqldict['Fax'].append(fax_)
                # sqldict['Website'].append(website_)
                # sqldict = bourange_same_length_array(sqldict)

                
            # name = table[i][-1].split('\n')[0]
            # address = table[-1][0].split('\n')[1:]
            # print()
            # print(name)
            # print()
            # if name.strip() != '':
            #     real_address_ = ' '.join(address)
            # print(real_address_)

ABI INSURANCE COMPANY LTD.
Redcliff Street, P. O. Box 2386, St. John's, Antigua
ANTIGUA INSURANCE CO. LTD. (ANICOL)
Long Street, St. John's, Antigua
CARIBBEAN ALLIANCE INSURANCE CO. LTD.
Caribbean Alliance House Cnr Newgate & Cross Streets, P.O. Box 1609, St. John’s, Antigua
C.G.I CONSUMERS GUARANTEE INSURANCE
COMPANY LIMITED Hill Park, Building 1 Unit 3, Friars Hill Road, St. John’s, Antigua.
C G UNITED INSURANCE LTD. (Formally) MASSY
UNITED INSURANCE LTD. Woods Centre, Friars Hill Road St. John's, Antigua
CUNA CARIBBEAN INSURANCE OECS LIMITED
Unit 3B, Green Gables Commercial Centre Mahogany Drive St. John’s Antigua
GENERAL INSURANCE COMPANY LTD.
Redcliffe Street, St. John's, Antigua
GK INSURANCE (EASTERN CARIBBEAN) LIMITED
Unit 2B, Green Gables Commercial Centre Mahogany Drive Gambles Development St. John’s, Antigua
GK Life INSURANCE EASTERN CARIBBEAN LIMITED
(Formally) SCOTIA INSURANCE EASTERN CARIBBEAN LIMITED Unit 2B, Green Gables Commercial Centre, Mahogany Drive, Gambles Develop